In [10]:
import os
import numpy as np
import pandas as pd
import joblib
from scipy.spatial.distance import mahalanobis

np.random.seed(42)

PROCESSED_DIR = r"C:\Project_EquiGuard\data\processed"
MODELS_DIR    = r"C:\Project_EquiGuard\outputs\models"
ARRAYS_DIR    = os.path.join(PROCESSED_DIR, "arrays")

print("Libraries loaded.")

Libraries loaded.


In [11]:
#Load arrays and feature names

X_train = np.load(os.path.join(ARRAYS_DIR, "X_train.npy"))
y_train = np.load(os.path.join(ARRAYS_DIR, "y_train.npy"))
X_val   = np.load(os.path.join(ARRAYS_DIR, "X_val.npy"))
y_val   = np.load(os.path.join(ARRAYS_DIR, "y_val.npy"))
X_test  = np.load(os.path.join(ARRAYS_DIR, "X_test.npy"))
y_test  = np.load(os.path.join(ARRAYS_DIR, "y_test.npy"))

feature_names = joblib.load(os.path.join(MODELS_DIR, "feature_names.pkl"))

print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"X_test:  {X_test.shape}")
print(f"Features: {len(feature_names)}")

X_train: (210, 28)
X_val:   (132, 28)
X_test:  (133, 28)
Features: 28


In [12]:
# Build healthy baseline from sound horses

sound_mask  = y_train == 0
X_sound     = X_train[sound_mask]

print(f"Sound horse rows in training: {X_sound.shape[0]}")
print(f"These form the healthy population baseline for Mahalanobis distance")

Sound horse rows in training: 42
These form the healthy population baseline for Mahalanobis distance


In [13]:
# Estimate healthy population distribution

mean_vec = np.mean(X_sound, axis=0)
cov_mat  = np.cov(X_sound, rowvar=False)

# Compute inverse covariance matrix
# Use pseudo-inverse as fallback if matrix is singular
try:
    inv_cov = np.linalg.inv(cov_mat)
    print("Covariance matrix inverted successfully.")
except np.linalg.LinAlgError:
    inv_cov = np.linalg.pinv(cov_mat)
    print("Singular matrix: used pseudo-inverse as fallback.")

print(f"Mean vector shape:      {mean_vec.shape}")
print(f"Covariance matrix shape: {cov_mat.shape}")
print(f"Inverse cov shape:       {inv_cov.shape}")

joblib.dump(mean_vec, os.path.join(MODELS_DIR, "mahal_mean.pkl"))
joblib.dump(inv_cov,  os.path.join(MODELS_DIR, "mahal_inv_cov.pkl"))
print("\nSaved: mahal_mean.pkl, mahal_inv_cov.pkl")

Singular matrix: used pseudo-inverse as fallback.
Mean vector shape:      (28,)
Covariance matrix shape: (28, 28)
Inverse cov shape:       (28, 28)

Saved: mahal_mean.pkl, mahal_inv_cov.pkl


In [14]:
# Compute Mahalanobis distance scores
def compute_mahalanobis(X, mean_vec, inv_cov):
    distances = []
    for row in X:
        try:
            d = mahalanobis(row, mean_vec, inv_cov)
        except ValueError:
            d = np.nan
        distances.append(d)
    return np.array(distances)

train_mahal = compute_mahalanobis(X_train, mean_vec, inv_cov)
val_mahal   = compute_mahalanobis(X_val,   mean_vec, inv_cov)
test_mahal  = compute_mahalanobis(X_test,  mean_vec, inv_cov)

print("Mahalanobis distances computed.")
print(f"\nTrain — mean distance (sound):     {train_mahal[y_train==0].mean():.3f}")
print(f"Train — mean distance (lame):      {train_mahal[y_train==1].mean():.3f}")
print(f"\nVal   — mean distance (sound):     {val_mahal[y_val==0].mean():.3f}")
print(f"Val   — mean distance (lame):      {val_mahal[y_val==1].mean():.3f}")

if train_mahal[y_train==1].mean() > train_mahal[y_train==0].mean():
    print("\nCheck passed: lame horses score higher than sound horses.")
else:
    print("\nWARNING: lame horses do NOT score higher — investigate.")

Mahalanobis distances computed.

Train — mean distance (sound):     4.800
Train — mean distance (lame):      8.614

Val   — mean distance (sound):     7.250
Val   — mean distance (lame):      10.816

Check passed: lame horses score higher than sound horses.


In [15]:
# Append Mahalanobis score as a new feature
X_train_final = np.column_stack([X_train, train_mahal])
X_val_final   = np.column_stack([X_val,   val_mahal])
X_test_final  = np.column_stack([X_test,  test_mahal])

print(f"X_train with Mahalanobis: {X_train_final.shape}")
print(f"X_val   with Mahalanobis: {X_val_final.shape}")
print(f"X_test  with Mahalanobis: {X_test_final.shape}")

# Save final arrays
np.save(os.path.join(ARRAYS_DIR, "X_train_final.npy"), X_train_final)
np.save(os.path.join(ARRAYS_DIR, "X_val_final.npy"),   X_val_final)
np.save(os.path.join(ARRAYS_DIR, "X_test_final.npy"),  X_test_final)

# Update feature names
feature_names_final = feature_names + ["mahalanobis_score"]
joblib.dump(feature_names_final, os.path.join(MODELS_DIR, "feature_names_final.pkl"))

print("\nAll final arrays saved.")
print(f"Total features including Mahalanobis: {len(feature_names_final)}")

X_train with Mahalanobis: (210, 29)
X_val   with Mahalanobis: (132, 29)
X_test  with Mahalanobis: (133, 29)

All final arrays saved.
Total features including Mahalanobis: 29
